# Lab 10 — DATS 6103

**Name:**

This notebook mirrors the lab page. Everything here also runs in the browser at
<https://www.smajhi.com/DATS-6103/labs/tree.html>, where the self-checks and the timers live; work in whichever you
prefer. The discussion half of the lab is not in this file — it happens away
from the keyboard.

**Conditions.** No AI assistants. The official Python, NumPy, Pandas,
Matplotlib and scikit-learn documentation and the course notes may be consulted
freely.

Run the self-check under each answer. It tells you whether your answer is right
without telling you what the answer is.

## Part A—Work it out

**Not submitted, and not written at the keyboard.** Work these in your team, on paper, one problem at a time; each is reported to the class as you finish it. The hint cards are on the lab page: <https://www.smajhi.com/DATS-6103/labs/tree.html>.

### A1. Two splits, one error rate

*Suggested: 5 minutes.*

A node holds 800 loan applications: 400 repaid and 400 defaulted. Two candidate
splits:

- **Split A** sends (300 repaid, 100 defaulted) left and (100, 300) right.
- **Split B** sends (200, 400) left and (200, 0) right.

a. Each child predicts its majority class. Compute the misclassification rate of
   each split.
b. Compute the weighted Gini impurity of each split. Which does a tree choose?
c. Why is that the better choice, even though the error rates agree?

### A2. Reading a fitted tree

*Suggested: 6 minutes.*

This tree was fitted, with depth 2, to 124 Italian wines from three cultivars,
labeled 0, 1 and 2, using 13 chemical measurements. The counts at each leaf are
the training wines of class 0, 1 and 2 that end there; at each split, the first
branch is "yes".

```
proline <= 755
├── color_intensity <= 4.90
│   ├── [ 1, 48,  3]   → class 1
│   └── [ 0,  1, 26]   → class 2
└── od280 <= 2.60
    ├── [ 0,  0,  4]   → class 2
    └── [40,  1,  0]   → class 0
```

a. Classify wine P (proline 1,050, color intensity 5.6, od280 3.1) and wine Q
   (proline 520, color intensity 7.2, od280 1.6).
b. What is the tree's training accuracy?
c. Flavanoids, another of the 13 measurements, does not appear in the tree, so
   its feature importance is exactly 0. A colleague concludes that flavanoids
   tell you nothing about the cultivar. What would you say?

### A3. The diagonal

*Suggested: 5 minutes.*

Points fall uniformly in the unit square, and a point is class 1 exactly when
$x_2 > x_1$: the boundary is the diagonal.

a. A tree with one split. Where does it split, and what is its accuracy?
b. A deeper tree approximates the diagonal by a staircase of $k$ equal steps.
   The points it misclassifies fill $k$ small triangles. Find its error as a
   function of $k$. How many steps does 99% accuracy need?
c. Logistic regression on $x_1, x_2$ gets this exactly. What single new feature
   would let a tree get it exactly with one split?

### A4. Averaging trees that agree

*Suggested: 6 minutes.*

The reading averaged independent estimates. Trees are not independent. If each of
$B$ trees has variance $\sigma^2$ and every pair has correlation $\rho$, their
average has variance
$$
\rho\,\sigma^2 + \frac{1 - \rho}{B}\,\sigma^2.
$$

a. Take $\sigma^2 = 1$ and $\rho = 0.6$, as for bagged trees that all split on
   the same strong feature. Compute the variance for $B = 10$, $100$, and as
   $B \to \infty$.
b. Repeat with $\rho = 0.2$.
c. Hiding features makes each tree a little worse: say $\sigma^2$ rises to 1.2,
   while $\rho$ falls from 0.6 to 0.2. With 100 trees, is it worth it?
d. Why can adding trees never make a forest worse, and what limits how much it
   helps?

## Part B—Build

### B1. How deep?

*Suggested: 10 minutes.*

`wn` is the wine data from A2: 178 wines, 13 measurements, three cultivars. For
each depth in `depths` (`None` means grow until every leaf is pure), record the
**training** accuracy and the **5-fold cross-validated** accuracy on the training
data, using `folds`. Then refit the best depth by cross-validation and score it
on the test set.

- `train_acc`, `cv_acc`---lists, one entry per depth;
- `best_depth`---the depth with the highest cross-validated accuracy;
- `test_acc`---that tree's accuracy on the test set.

**Already loaded:** `wn`, `X_train`, `X_test`, `y_train` and `y_test`, `folds`,
`depths`, and `tree_at(depth)`, which returns a new, unfitted
tree of that depth.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
wn = load_wine()
X, y = wn.data, wn.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=6103, stratify=y)
folds = StratifiedKFold(5, shuffle=True, random_state=6103)
depths = [1, 2, 3, 4, 5, 6, None]
def tree_at(depth):
    return DecisionTreeClassifier(max_depth=depth, random_state=0)
def show(v):
    """How a value prints: a blank as 'blank', a number to three decimals."""
    if v is ...:
        return "blank"
    if isinstance(v, (float, np.floating)):
        return f"{v:.3f}"
    if isinstance(v, dict):
        return "{" + ", ".join(f"{k}: {show(x)}" for k, x in v.items()) + "}"
    if isinstance(v, (list, tuple, np.ndarray)):
        return "[" + ", ".join(show(x) for x in v) + "]"
    return str(v)

In [ ]:
train_acc, cv_acc = [], []
for d in depths:
    clf = tree_at(d)
    train_acc.append(...)
    cv_acc.append(...)
best_depth = ...
test_acc = ...

print(f"{'depth':>6} {'train':>7} {'cv':>7}")
for d, a, c in zip(depths, train_acc, cv_acc):
    print(f"{str(d):>6} {show(a):>7} {show(c):>7}")
print("best depth:", show(best_depth), "| test accuracy:", show(test_acc))

In [ ]:
# self-check: run this after your answer above
wt, wc = [], []
for d_ in depths:
    c_ = DecisionTreeClassifier(max_depth=d_, random_state=0)
    wt.append(c_.fit(X_train, y_train).score(X_train, y_train))
    wc.append(cross_val_score(c_, X_train, y_train, cv=folds).mean())
wb = depths[int(np.argmax(wc))]
wtest = DecisionTreeClassifier(max_depth=wb, random_state=0).fit(X_train, y_train).score(X_test, y_test)
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("train_acc", train_acc, lambda: np.allclose(train_acc, wt),
          "should be each tree's accuracy on the data it was fit on, one entry per depth."),
    _part("cv_acc", cv_acc, lambda: np.allclose(cv_acc, wc),
          "should be cross_val_score(clf, X_train, y_train, cv=folds).mean(), one entry per depth."),
    _part("best_depth", best_depth, lambda: best_depth == wb,
          "should maximize the cross-validated accuracy, not the training accuracy."),
    _part("test_acc", test_acc, lambda: abs(float(test_acc) - wtest) < 1e-9,
          "should be the best_depth tree, refit on all of X_train and scored on X_test."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

### B2. What the tree used

*Suggested: 12 minutes.*

`tree2` is the depth-2 tree from A2, fitted for you, and `t` is its internal
structure. Nodes are numbered from 0, the root, and for node `i`:
`t.feature[i]` and `t.threshold[i]` are its split, `t.impurity[i]` its Gini,
`t.n_node_samples[i]` how many training wines reach it, and
`t.children_left[i]`, `t.children_right[i]` its children (both `-1` at a leaf).

- `root_feature`, `root_threshold`---the root's feature, by **name**, and
  threshold;
- `decrease`---for each internal node in node order, the reading's
  ${\tt Imp}(j)$: the impurity removed by its split, weighted by the share of
  **all** the training wines that reach each node;
- `imp_root`---the root's share of the total decrease. Compare it with
  `tree2.feature_importances_`;
- `flav_cv`, `proline_cv`---the 5-fold cross-validated accuracy of a depth-2
  tree that sees **only** flavanoids, and of one that sees only proline.

**Already loaded:** `wn`, `X_train`, `y_train`, `folds`, `tree_at`, `tree2`, `t`,
and the column numbers `flav` and `proline`.

In [ ]:
tree2 = tree_at(2).fit(X_train, y_train)
t = tree2.tree_
flav = wn.feature_names.index("flavanoids")
proline = wn.feature_names.index("proline")

In [ ]:
root_feature = ...
root_threshold = ...

n, g = t.n_node_samples, t.impurity
decrease = []
for i in range(t.node_count):
    L, R = t.children_left[i], t.children_right[i]
    if L != -1:                                # an internal node
        decrease.append(...)
imp_root = ...

flav_cv = ...
proline_cv = ...

print("root:", show(root_feature), "at", show(root_threshold))
print("decrease:", show(decrease))
print("root's share:", show(imp_root), "| feature_importances_ for proline:",
      show(tree2.feature_importances_[proline]))
print("flavanoids alone:", show(flav_cv), "| proline alone:", show(proline_cv))

In [ ]:
# self-check: run this after your answer above
n_, g_ = t.n_node_samples, t.impurity
wd, wd_own = [], []
for i_ in range(t.node_count):
    L_, R_ = t.children_left[i_], t.children_right[i_]
    if L_ != -1:
        wd.append((n_[i_] * g_[i_] - n_[L_] * g_[L_] - n_[R_] * g_[R_]) / n_[0])
        wd_own.append((n_[i_] * g_[i_] - n_[L_] * g_[L_] - n_[R_] * g_[R_]) / n_[i_])
wf = cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=0), X_train[:, [flav]], y_train, cv=folds).mean()
wp = cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=0), X_train[:, [proline]], y_train, cv=folds).mean()
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("root_feature", root_feature, lambda: str(root_feature) == wn.feature_names[t.feature[0]],
          "that is the column number; look its name up in wn.feature_names."
          if not _blank(root_feature) and isinstance(root_feature, (int, np.integer))
          else "should be the name of the feature node 0 splits on."),
    _part("root_threshold", root_threshold, lambda: abs(float(root_threshold) - float(t.threshold[0])) < 1e-6,
          "should be node 0's threshold."),
    _part("decrease", decrease, lambda: np.allclose(decrease, wd),
          "that divides by the rows reaching each node; the reading divides by ALL the training rows, n[0]."
          if not _blank(decrease) and len(decrease) == len(wd_own) and np.allclose(decrease, wd_own)
          else "should be (n[i]*g[i] - n[L]*g[L] - n[R]*g[R]) / n[0] for each internal node."),
    _part("imp_root", imp_root, lambda: abs(float(imp_root) - wd[0] / sum(wd)) < 1e-9,
          "should be decrease[0] divided by the sum of decrease."),
    _part("flav_cv", flav_cv, lambda: abs(float(flav_cv) - wf) < 1e-9,
          "should be cross_val_score on X_train[:, [flav]], with a depth-2 tree and folds."),
    _part("proline_cv", proline_cv, lambda: abs(float(proline_cv) - wp) < 1e-9,
          "should be cross_val_score on X_train[:, [proline]], with a depth-2 tree and folds."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

### B3. Ten resamples, ten trees

*Suggested: 10 minutes.*

Back to last week's breast cancer data: 569 tumors, 30 measurements, `0` for
**malignant**. `boots` holds ten bootstrap samples of the training rows, as row
numbers. Grow a full tree on each and record:

- `roots`---the name of each tree's root feature;
- `accs`---each tree's test accuracy;
- `n_roots`---how many different root features appear;
- `spread`---the best test accuracy minus the worst.

Then bag 200 trees and report:

- `bag_oob`---its out-of-bag accuracy;
- `bag_test`---its test accuracy.

**Already loaded:** `bc`, `X_train`, `X_test`, `y_train` and `y_test`, `boots`,
and `BaggingClassifier`.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
bc = load_breast_cancer()
X, y = bc.data, bc.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=6103, stratify=y)
rng = np.random.default_rng(6103)
boots = [rng.integers(0, len(y_train), len(y_train)) for _ in range(10)]

In [ ]:
roots, accs = [], []
for b in boots:
    tr = DecisionTreeClassifier(random_state=0).fit(X_train[b], y_train[b])
    roots.append(...)
    accs.append(...)
n_roots = ...
spread = ...

bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=200,
                        oob_score=True, random_state=0)
bag_oob = ...
bag_test = ...

for r, a in zip(roots, accs):
    print(f"{show(r):>24}  {show(a)}")
print("different roots:", show(n_roots), "| spread:", show(spread))
print("bagged, out of bag:", show(bag_oob), "| bagged, test:", show(bag_test))

In [ ]:
# self-check: run this after your answer above
wr, wa = [], []
for b_ in boots:
    tr_ = DecisionTreeClassifier(random_state=0).fit(X_train[b_], y_train[b_])
    wr.append(str(bc.feature_names[tr_.tree_.feature[0]]))
    wa.append(tr_.score(X_test, y_test))
bg_ = BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, oob_score=True, random_state=0).fit(X_train, y_train)
wo, wtest = bg_.oob_score_, bg_.score(X_test, y_test)
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("roots", roots, lambda: [str(r) for r in roots] == wr,
          "should be names: look each tree's tr.tree_.feature[0] up in bc.feature_names."
          if not _blank(roots) and len(roots) > 0 and isinstance(roots[0], (int, np.integer))
          else "should be the name of each tree's root feature, one per bootstrap sample."),
    _part("accs", accs, lambda: np.allclose(accs, wa), "should be each tree's accuracy on X_test."),
    _part("n_roots", n_roots, lambda: int(n_roots) == len(set(wr)), "should count the distinct names in roots."),
    _part("spread", spread, lambda: abs(float(spread) - (max(wa) - min(wa))) < 1e-9,
          "should be max(accs) - min(accs)."),
    _part("bag_oob", bag_oob, lambda: abs(float(bag_oob) - wo) < 1e-9,
          "should be bag.oob_score_ after fitting bag on the training data."),
    _part("bag_test", bag_test, lambda: abs(float(bag_test) - wtest) < 1e-9,
          "should be the fitted bag's accuracy on X_test."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

### B4. The final

*Suggested: 14 minutes.*

Four candidates for the breast cancer data: the best single tree from a depth
search like B1's, a random forest, gradient boosting with shallow trees, and
last week's scaled logistic regression. Choose one **by 5-fold cross-validation
on the training data**, then touch the test set once.

- `cv`---a dict from each candidate's name to its cross-validated accuracy;
- `winner`---the name with the highest;
- `final`---that candidate, refit on all the training data;
- `test_acc`---the winner, refit on all the training data, scored on the test
  set.

**Already loaded:** `X_train`, `X_test`, `y_train` and `y_test`, `folds`, and
`candidates`, a dict of four unfitted models.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
candidates = {
    "tree":     DecisionTreeClassifier(max_depth=3, random_state=0),
    "forest":   RandomForestClassifier(n_estimators=200, max_features=2, random_state=0),
    "boosting": GradientBoostingClassifier(n_estimators=100, max_depth=2, random_state=0),
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
}

In [ ]:
cv = {}
for name, model in candidates.items():
    cv[name] = ...
winner = ...
final = ...
y_hat = ...
test_acc = ...

print("cross-validated:", show(cv))
print("winner:", show(winner), "| test accuracy:", show(test_acc))

In [ ]:
# self-check: run this after your answer above
from sklearn.base import clone
wc = {}
for k_, m_ in candidates.items():
    wc[k_] = cross_val_score(clone(m_), X_train, y_train, cv=folds).mean()
ww = max(wc, key=wc.get)
yh_ = clone(candidates[ww]).fit(X_train, y_train).predict(X_test)
wtest = float((yh_ == y_test).mean())
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("cv", cv, lambda: set(cv) == set(wc) and all(abs(float(cv[k]) - wc[k]) < 1e-9 for k in wc),
          "cv[name] should be cross_val_score(model, X_train, y_train, cv=folds).mean(), for every candidate."),
    _part("winner", winner, lambda: winner == ww, "should be the name with the highest cross-validated accuracy."),
    _part("test_acc", test_acc, lambda: abs(float(test_acc) - wtest) < 1e-9,
          "should be the winner's accuracy on X_test."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

## Before you leave

Upload this notebook to Blackboard. It is the record of what you did; the lab
grade itself comes from what you do in the room.

Then post a comment at the bottom of the lab page, <https://www.smajhi.com/DATS-6103/labs/tree.html>. One line is enough.
It proves your GitHub account works with the comment system, which is how every
week's reading credit is earned.